# Repair Author Sequence Bug

One-time data repair for the author sequence mismatch in `work_authors`.

## Problem
`CreateWorkAuthors` joined `openalex_works_base` author positions to `works_legacy.work_authors` on `(work_id, author_sequence)`, but provenance-based ordering in `openalex_works_base` can differ from the legacy POSEXPLODE ordering. This misaligned ~36.5M author_ids across ~19.9M works.

## Scoping
Affected rows are identified by comparing the assigned author entity's parsed last name to the raw_author_name's parsed last name. A mismatch means the wrong author was assigned to that position.

## Fix Strategy
1. Identify affected rows via parsed last name mismatch (author entity vs raw_author_name)
2. For affected works, re-match to `works_legacy.work_authors` by name (exact + parsed fallback)
3. Only NULL unmatched rows with additional evidence of being wrong (duplicate author_id, ORCID conflict)
4. Queue affected works for pipeline reprocessing

## Prerequisites
- Run on X-Large SQL warehouse (`3996dc0a9b183ce3`)
- Code fixes already deployed: block_key (b4748dc), COALESCE (07931d2)
- `CreateParsedNamesLookup` must be run first to ensure all names are in parsed_names_lookup (63d8456)

### Cell 1: Diagnostic — Quantify affected rows

Count work_authors rows where the assigned author entity's parsed last name differs from the raw_author_name's parsed last name.

In [ ]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT wa.work_id) AS work_count
FROM openalex.works.work_authors wa
INNER JOIN openalex.authors.openalex_authors oa
    ON wa.author_id = oa.id
INNER JOIN openalex.authors.parsed_names_lookup pn_raw
    ON TRIM(wa.raw_author_name) = pn_raw.raw_author_name
INNER JOIN openalex.authors.parsed_names_lookup pn_display
    ON TRIM(oa.display_name) = pn_display.raw_author_name
WHERE wa.work_id <= 7000000000
  AND wa.author_id IS NOT NULL
  AND pn_raw.parsed_name.last IS NOT NULL AND pn_raw.parsed_name.last != ''
  AND pn_display.parsed_name.last IS NOT NULL AND pn_display.parsed_name.last != ''
  AND pn_raw.parsed_name.last != pn_display.parsed_name.last
  AND pn_display.parsed_name.last NOT IN (pn_raw.parsed_name.first, pn_raw.parsed_name.middle, pn_raw.parsed_name.last)
  AND pn_raw.parsed_name.last NOT IN (pn_display.parsed_name.first, pn_display.parsed_name.middle, pn_display.parsed_name.last)

### Cell 2: Identify affected work_ids

Find works where at least one author position has a parsed last name mismatch between the raw_author_name and the assigned author entity's display_name.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_sequence_repair_work_ids AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
INNER JOIN openalex.authors.openalex_authors oa
    ON wa.author_id = oa.id
INNER JOIN openalex.authors.parsed_names_lookup pn_raw
    ON TRIM(wa.raw_author_name) = pn_raw.raw_author_name
INNER JOIN openalex.authors.parsed_names_lookup pn_display
    ON TRIM(oa.display_name) = pn_display.raw_author_name
WHERE wa.work_id <= 7000000000
  AND wa.author_id IS NOT NULL
  AND pn_raw.parsed_name.last IS NOT NULL AND pn_raw.parsed_name.last != ''
  AND pn_display.parsed_name.last IS NOT NULL AND pn_display.parsed_name.last != ''
  AND pn_raw.parsed_name.last != pn_display.parsed_name.last
  AND pn_display.parsed_name.last NOT IN (pn_raw.parsed_name.first, pn_raw.parsed_name.middle, pn_raw.parsed_name.last)
  AND pn_raw.parsed_name.last NOT IN (pn_display.parsed_name.first, pn_display.parsed_name.middle, pn_display.parsed_name.last)

### Cell 3: Build repair table — Pass 1 (exact name match)

For affected works only, join `work_authors` to backfill on exact normalized `raw_author_name`. Uses `ROW_NUMBER` tiebreaker for duplicate names on the same paper.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_sequence_repair AS
WITH legacy_wa AS (
    SELECT
        work_id,
        raw_author_name AS legacy_name,
        LOWER(TRIM(raw_author_name)) AS legacy_name_norm,
        author_id AS legacy_author_id,
        author_sequence AS legacy_sequence
    FROM openalex.works_legacy.work_authors
    WHERE author_id IS NOT NULL
      AND work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
),
current_wa AS (
    SELECT
        work_id,
        author_sequence,
        author_id,
        raw_author_name,
        LOWER(TRIM(raw_author_name)) AS current_name_norm
    FROM openalex.works.work_authors
    WHERE work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
),
name_matched AS (
    SELECT
        cwa.work_id,
        cwa.author_sequence,
        cwa.author_id AS current_author_id,
        cwa.raw_author_name,
        lwa.legacy_author_id,
        lwa.legacy_sequence,
        ROW_NUMBER() OVER (
            PARTITION BY cwa.work_id, cwa.current_name_norm
            ORDER BY cwa.author_sequence
        ) AS current_rank,
        ROW_NUMBER() OVER (
            PARTITION BY lwa.work_id, lwa.legacy_name_norm
            ORDER BY lwa.legacy_sequence
        ) AS legacy_rank
    FROM current_wa cwa
    INNER JOIN legacy_wa lwa
        ON cwa.work_id = lwa.work_id
        AND cwa.current_name_norm = lwa.legacy_name_norm
)
SELECT
    work_id,
    author_sequence,
    raw_author_name,
    current_author_id,
    legacy_author_id AS backfill_author_id,
    'EXACT' AS match_type,
    CASE
        WHEN current_author_id = legacy_author_id THEN 'ALREADY_CORRECT'
        WHEN current_author_id IS NULL THEN 'WAS_NULL'
        ELSE 'WRONG_AUTHOR_ID'
    END AS repair_status
FROM name_matched
WHERE current_rank = legacy_rank

### Cell 3b: Build repair table — Pass 2 (parsed name fallback)

For affected work_authors rows not matched in Pass 1, join via `parsed_names_lookup` on `(work_id, parsed last, parsed first)`. This catches provenance-driven name variations like accent differences, period handling, and initial formatting.

In [ ]:
INSERT INTO openalex.authors.author_sequence_repair
WITH legacy_wa AS (
    SELECT
        work_id,
        raw_author_name AS legacy_name,
        author_id AS legacy_author_id,
        author_sequence AS legacy_sequence
    FROM openalex.works_legacy.work_authors
    WHERE author_id IS NOT NULL
      AND work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
),
-- Legacy names enriched with parsed components
legacy_parsed AS (
    SELECT
        lwa.*,
        pn.parsed_name.last AS legacy_last,
        pn.parsed_name.first AS legacy_first
    FROM legacy_wa lwa
    INNER JOIN openalex.authors.parsed_names_lookup pn
        ON TRIM(lwa.legacy_name) = pn.raw_author_name
    WHERE pn.parsed_name.last IS NOT NULL AND pn.parsed_name.last != ''
),
-- Unmatched work_authors rows from Pass 1
unmatched_wa AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name
    FROM openalex.works.work_authors wa
    LEFT JOIN openalex.authors.author_sequence_repair r
        ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
    WHERE wa.work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
      AND r.work_id IS NULL
),
-- Unmatched rows enriched with parsed components
unmatched_parsed AS (
    SELECT
        uwa.*,
        pn.parsed_name.last AS current_last,
        pn.parsed_name.first AS current_first
    FROM unmatched_wa uwa
    INNER JOIN openalex.authors.parsed_names_lookup pn
        ON TRIM(uwa.raw_author_name) = pn.raw_author_name
    WHERE pn.parsed_name.last IS NOT NULL AND pn.parsed_name.last != ''
),
-- Join on parsed name components
parsed_matched AS (
    SELECT
        up.work_id,
        up.author_sequence,
        up.author_id AS current_author_id,
        up.raw_author_name,
        lp.legacy_author_id,
        lp.legacy_sequence,
        ROW_NUMBER() OVER (
            PARTITION BY up.work_id, up.current_last, up.current_first
            ORDER BY up.author_sequence
        ) AS current_rank,
        ROW_NUMBER() OVER (
            PARTITION BY lp.work_id, lp.legacy_last, lp.legacy_first
            ORDER BY lp.legacy_sequence
        ) AS legacy_rank
    FROM unmatched_parsed up
    INNER JOIN legacy_parsed lp
        ON up.work_id = lp.work_id
        AND up.current_last = lp.legacy_last
        AND up.current_first = lp.legacy_first
)
SELECT
    work_id,
    author_sequence,
    raw_author_name,
    current_author_id,
    legacy_author_id AS backfill_author_id,
    'PARSED' AS match_type,
    CASE
        WHEN current_author_id = legacy_author_id THEN 'ALREADY_CORRECT'
        WHEN current_author_id IS NULL THEN 'WAS_NULL'
        ELSE 'WRONG_AUTHOR_ID'
    END AS repair_status
FROM parsed_matched
WHERE current_rank = legacy_rank

### Cell 3c: Diagnostic — Analyze repair coverage

Review repair_status and match_type distribution, plus count of unmatched rows still in affected works.

In [ ]:
SELECT match_type, repair_status, COUNT(*) AS row_count, COUNT(DISTINCT work_id) AS work_count
FROM openalex.authors.author_sequence_repair
GROUP BY match_type, repair_status

UNION ALL

SELECT
    'N/A' AS match_type,
    'UNMATCHED' AS repair_status,
    COUNT(*) AS row_count,
    COUNT(DISTINCT wa.work_id) AS work_count
FROM openalex.works.work_authors wa
LEFT JOIN openalex.authors.author_sequence_repair r
    ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
WHERE wa.work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
  AND r.work_id IS NULL

### Cell 4: Apply repair via MERGE

Restore correct author_ids from backfill ground truth.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT work_id, author_sequence, backfill_author_id
    FROM openalex.authors.author_sequence_repair
    WHERE repair_status IN ('WRONG_AUTHOR_ID', 'WAS_NULL')
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN UPDATE SET
    target.author_id = source.backfill_author_id,
    target.updated_at = current_timestamp()

### Cell 5: NULL high-confidence wrong matches in unmatched legacy rows

For legacy work_authors rows still unmatched after both passes, only NULL author_ids where we have additional evidence they are wrong: duplicate author_id on the same work, or ORCID conflict. Rows without such evidence keep their existing author_id.

In [ ]:
UPDATE openalex.works.work_authors wa
SET wa.author_id = NULL,
    wa.updated_at = current_timestamp()
WHERE wa.work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
  AND wa.author_id IS NOT NULL
  AND NOT EXISTS (
      SELECT 1 FROM openalex.authors.author_sequence_repair r
      WHERE r.work_id = wa.work_id
        AND r.author_sequence = wa.author_sequence
  )
  AND (
      -- Duplicate author_id on same work
      EXISTS (
          SELECT 1
          FROM openalex.works.work_authors wa2
          WHERE wa2.work_id = wa.work_id
            AND wa2.author_id = wa.author_id
            AND wa2.author_sequence != wa.author_sequence
      )
      OR
      -- ORCID conflict
      EXISTS (
          SELECT 1
          FROM (
              SELECT id AS work_id, pos AS author_sequence, auth.author.orcid AS raw_orcid
              FROM openalex.works.openalex_works_base
              LATERAL VIEW POSEXPLODE(authorships) t AS pos, auth
              WHERE auth.author.orcid IS NOT NULL AND auth.author.orcid != ''
          ) wb
          JOIN openalex.authors.openalex_authors oa
              ON oa.id = wa.author_id
          WHERE wb.work_id = wa.work_id
            AND wb.author_sequence = wa.author_sequence
            AND oa.orcid IS NOT NULL
            AND oa.orcid != wb.raw_orcid
      )
  )

### Cell 6b: Handle post-backfill works — Duplicate author_ids

NULL author_ids where the same author_id appears at multiple positions on one work.

In [ ]:
UPDATE openalex.works.work_authors wa
SET wa.author_id = NULL,
    wa.updated_at = current_timestamp()
WHERE wa.work_id > 7000000000
  AND wa.author_id IS NOT NULL
  AND EXISTS (
      SELECT 1
      FROM openalex.works.work_authors wa2
      WHERE wa2.work_id = wa.work_id
        AND wa2.author_id = wa.author_id
        AND wa2.author_sequence != wa.author_sequence
  )

### Cell 7: Queue affected works for UpdateWorkAuthorships

Insert repaired work_ids into `curated_work_ids_pending_sync` so the next E2E run propagates fixes.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync (work_id, curated_ras, added_datetime)
SELECT DISTINCT
    work_id,
    'author_sequence_repair' AS curated_ras,
    current_timestamp() AS added_datetime
FROM (
    -- Legacy works repaired from backfill
    SELECT work_id
    FROM openalex.authors.author_sequence_repair
    WHERE repair_status IN ('WRONG_AUTHOR_ID', 'WAS_NULL')

    UNION

    -- Legacy works with unmatched names (NULLed in Cell 5)
    SELECT wa.work_id
    FROM openalex.works.work_authors wa
    WHERE wa.work_id <= 7000000000
      AND wa.author_id IS NULL
      AND wa.updated_at >= current_timestamp() - INTERVAL 1 HOUR

    UNION

    -- Post-backfill works NULLed in Cells 6a/6b
    SELECT wa.work_id
    FROM openalex.works.work_authors wa
    WHERE wa.work_id > 7000000000
      AND wa.author_id IS NULL
      AND wa.updated_at >= current_timestamp() - INTERVAL 1 HOUR
) affected
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.institutions.curated_work_ids_pending_sync p
    WHERE p.work_id = affected.work_id
)

### Cell 8: Final verification

Sample repaired rows to confirm author_ids now match backfill.

In [ ]:
SELECT
    wa.work_id,
    wa.author_sequence,
    wa.author_id AS new_author_id,
    wa.raw_author_name,
    r.current_author_id AS old_author_id,
    r.backfill_author_id,
    r.repair_status
FROM openalex.works.work_authors wa
INNER JOIN openalex.authors.author_sequence_repair r
    ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
WHERE r.repair_status = 'WRONG_AUTHOR_ID'
LIMIT 100